# 03 — Baseline Structural Analysis (Go/No-Go gate)

Loads `stats_base.json` and evaluates whether the base-model structural
statistics show enough prompt-level variation to justify the LoRA fine-tuning
experiment.

**Go/No-Go criterion (both must hold):**
1. At least **2** structural metrics have IQR > 0 and non-collapsed distributions.
2. A logistic regression classifier on structural metrics achieves **above-majority accuracy** for `correct vs incorrect` in cross-validation.

The final cell prints a clear GO / STOP recommendation.

**No GPU required.** Run locally.

## 0 — Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo.")

MY_WORK = _my_work if _root else Path(".").resolve()
STATS_FILE = MY_WORK / "results" / "statistics" / "stats_base.json"
print(f"Stats file: {STATS_FILE}")

## 1 — Load statistics

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from utils.graph_statistics import load_statistics, aggregate_statistics

all_stats = load_statistics(STATS_FILE)
stats = [s for s in all_stats if s.get("attribution_succeeded")]

print(f"Total entries  : {len(all_stats)}")
print(f"Succeeded      : {len(stats)}")
print(f"Success rate   : {len(stats)/len(all_stats):.1%}")
print()

agg = aggregate_statistics(all_stats)
metrics = ['n_active_features','edge_density','mean_top50_score',
           'top10_over_top50','layer_entropy','mean_error_node_weight']

print(f"{'Metric':<28} {'Mean':>10} {'Std':>10} {'Median':>10} {'IQR':>10}")
print("-" * 72)
for m in metrics:
    v = agg.get(m)
    if v:
        print(f"{m:<28} {v['mean']:>10.4f} {v['std']:>10.4f} {v['median']:>10.4f} {v['iqr']:>10.4f}")

## 2 — Distribution plots for each metric

Check that distributions are non-collapsed (IQR > 0).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for ax, metric in zip(axes, metrics):
    vals = [s[metric] for s in stats if s.get(metric) is not None]
    if not vals:
        ax.set_title(f"{metric}\n(no data)")
        continue
    arr = np.array(vals)
    # Split by correct vs incorrect first-token
    correct_vals   = [s[metric] for s in stats
                      if s.get(metric) is not None
                      and s.get('label') == (s.get('prob_true', 0) > s.get('prob_false', 0))]
    incorrect_vals = [s[metric] for s in stats
                      if s.get(metric) is not None
                      and s.get('label') != (s.get('prob_true', 0) > s.get('prob_false', 0))]

    ax.hist(arr, bins=20, alpha=0.6, color='steelblue', label='all')
    if correct_vals:
        ax.hist(correct_vals,   bins=20, alpha=0.5, color='green',  label='correct')
    if incorrect_vals:
        ax.hist(incorrect_vals, bins=20, alpha=0.5, color='red',    label='incorrect')
    q1, q3 = np.percentile(arr, [25, 75])
    ax.axvline(q1, color='gray', linestyle='--', linewidth=0.8)
    ax.axvline(q3, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(f"{metric}\nIQR={q3-q1:.4f}")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.suptitle("Base model: structural metric distributions", y=1.01, fontsize=12)

plot_path = MY_WORK / "results" / "statistics" / "base_metric_distributions.png"
plt.savefig(plot_path, bbox_inches="tight")
print(f"Saved: {plot_path}")
plt.show()

## 3 — Calibration: logit gap distribution by label

Shows whether the base model shows any separation between True and False prompts.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

gaps_true  = [s['logit_gap'] for s in stats if s.get('logit_gap') is not None and s['label']]
gaps_false = [s['logit_gap'] for s in stats if s.get('logit_gap') is not None and not s['label']]

if gaps_true:
    ax.hist(gaps_true,  bins=20, alpha=0.6, color='green', label='True prompts')
if gaps_false:
    ax.hist(gaps_false, bins=20, alpha=0.6, color='red',   label='False prompts')

ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
ax.set_xlabel("logit( True) − logit(False)")
ax.set_ylabel("Count")
ax.set_title("Calibration: logit gap by label (base model)")
ax.legend()
plt.tight_layout()
plt.savefig(MY_WORK / "results" / "statistics" / "base_logit_gap.png", bbox_inches="tight")
plt.show()

if gaps_true and gaps_false:
    from scipy import stats as scipy_stats
    stat_val, p_val = scipy_stats.mannwhitneyu(gaps_true, gaps_false, alternative='two-sided')
    print(f"Mann-Whitney U: U={stat_val:.1f}, p={p_val:.4f}")
    if p_val < 0.05:
        print("Base model shows statistically significant logit gap difference by label.")
    else:
        print("No significant logit gap difference by label (expected for base model).")

## 4 — Logistic regression: correct vs incorrect from structural metrics

Condition 2 of the Go/No-Go gate: does a simple linear classifier
using the structural metrics predict whether the model's first-token
prediction is correct?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

feature_cols = [
    'n_active_features', 'edge_density', 'mean_top50_score',
    'top10_over_top50', 'layer_entropy',
]

# Build feature matrix
rows = []
for s in stats:
    row = [s.get(c) for c in feature_cols]
    if any(v is None for v in row):
        continue
    # Label: is the model's first-token prediction correct?
    prob_true  = s.get('prob_true',  0) or 0
    prob_false = s.get('prob_false', 0) or 0
    pred_label = prob_true > prob_false
    correct = int(pred_label == s['label'])
    rows.append(row + [correct])

if not rows:
    print("No complete rows for classifier. Check that stats_base.json is populated.")
else:
    arr = np.array(rows)
    X = arr[:, :-1]
    y = arr[:, -1].astype(int)

    majority_acc = max(y.mean(), 1 - y.mean())

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=500, C=1.0)),
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')

    print(f"Feature matrix shape : {X.shape}")
    print(f"Class balance        : {y.mean():.1%} correct")
    print(f"Majority baseline    : {majority_acc:.1%}")
    print(f"CV accuracy (5-fold) : {cv_scores.mean():.1%} ± {cv_scores.std():.1%}")
    print()

    # Fit on all data to inspect coefficients
    pipe.fit(X, y)
    coefs = pipe.named_steps['clf'].coef_[0]
    print("Logistic regression coefficients (standardised features):")
    for feat, coef in sorted(zip(feature_cols, coefs), key=lambda x: -abs(x[1])):
        print(f"  {feat:<28} {coef:>+.4f}")

    logreg_beats_majority = cv_scores.mean() > majority_acc
    print()
    print(f"Classifier beats majority: {logreg_beats_majority}")

## 5 — IQR spread check (Condition 1)

In [ ]:
primary_metrics = [
    'n_active_features', 'edge_density', 'mean_top50_score',
    'top10_over_top50', 'layer_entropy',
]

non_trivial = []
for m in primary_metrics:
    vals = [s[m] for s in stats if s.get(m) is not None]
    if vals:
        arr = np.array(vals)
        q1, q3 = np.percentile(arr, [25, 75])
        iqr = q3 - q1
        status = "OK" if iqr > 0 else "COLLAPSED"
        print(f"  {m:<28} IQR={iqr:.6f}  [{status}]")
        if iqr > 0:
            non_trivial.append(m)

print()
print(f"Metrics with IQR > 0: {len(non_trivial)}/{len(primary_metrics)}")
condition1 = len(non_trivial) >= 2
print(f"Condition 1 (>=2 non-trivial metrics): {condition1}")

## 6 — GO / STOP decision

In [ ]:
condition2 = logreg_beats_majority if 'logreg_beats_majority' in dir() else False

print("=" * 60)
print("GO / NO-GO DECISION")
print("=" * 60)
print(f"Condition 1 — >=2 metrics with IQR>0 : {'PASS' if condition1 else 'FAIL'}")
print(f"Condition 2 — classifier > majority  : {'PASS' if condition2 else 'FAIL'}")
print()

if condition1 and condition2:
    print("DECISION: GO — proceed to 04_lora_finetuning_triangle.ipynb")
elif condition1 and not condition2:
    print("DECISION: CONDITIONAL GO")
    print("  Structural variation exists but classifier signal is weak.")
    print("  Proceed with caution and note limitation in thesis.")
else:
    print("DECISION: STOP")
    print("  Structural statistics show insufficient variation across prompts.")
    print("  The structural-signal hypothesis is not supported on this setup.")
    print("  Do not proceed to LoRA fine-tuning.")